# Fly-vs-Vol Screener — SOFR butterflies vs option-implied distributions

The fly and the options price the same random path. `fly_bp = 2*f_belly - f_front - f_back = D1 - D2`
is the risk-neutral **mean** of the fly settlement `phi`; the option RNDs carry its **shape**.
This screener strips out the part where the two markets agree by construction (the mean) and
compares what is left:

- **Tier 1 (marginal-only):** `fly_median_path = 2*med_belly - med_front - med_back`;
  `tail_rent = fly - fly_median_path` = the part of the fly paid for asymmetric tails.
- **Tier 2 (comonotone):** one policy factor `f_i = F_i^{-1}(U)` gives the full distribution of
  `phi`, the move-count table `P(N1-N2=j)`, and `prob_delta = P(N1>N2) - P(N1<N2)` vs the desk
  heuristic `fly/25`.
- **Tier 3 (couplings):** Gaussian copula on historical correlations; option-implied FOMC-path
  joint. Divergence from comonotone ~ reversal-risk premium.
- **Rich/cheap:** daily history + rolling z-scores of the gap series (persistent gap = premium;
  deviation = signal).

Run `notebooks/rv/run_fly_vs_vol_screener.py` first to build `notebooks/data/fly_vs_vol/`.
Spec: `docs/superpowers/specs/2026-07-28-fly-vs-vol-rv-framework-design.md`.

In [ ]:
import sys; sys.path.append("../../")

import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from MDP.STIRFutures.STIRFutureOptionMDP import STIRFutureOptionMDP
from RVUtils.FlyVsVol import (
    ContractMarginal, FlyDefinition, FlyVsVolConfig, adjacent_triples,
    build_fly_snapshot, comonotone_grid, gaussian_copula_sample, historical_corr,
    history_zscores, path_metrics, run_fly_screener, screener_table,
)
from RVUtils.FlyVsVol.plotting import (
    plot_fly_distribution, plot_history_panel, plot_move_table,
)
from RVUtils.ImpliedDistribution import SFRImpliedDistribution, resolve_strip_symbols

AS_OF = datetime.date(2026, 7, 27)
DATA_DIR = Path("../data/fly_vs_vol")
SYMBOLS = resolve_strip_symbols("2y", as_of=AS_OF)
TRIPLES = adjacent_triples(SYMBOLS)
config = FlyVsVolConfig()
print(SYMBOLS)
print([t.label for t in TRIPLES])

## 1. Snapshot monitor at the reference date

In [ ]:
mdp = STIRFutureOptionMDP(source="BARCHART_STIRFO-QL")
dist = SFRImpliedDistribution(anchor_wings=True)

marginals = {}
for sym in SYMBOLS:
    try:
        smile = mdp.fetch_sabr_smile({
            "symbol": sym, "as_of": AS_OF, "strike_offsets_bps": "listed",
        })
        snap = dist.extract(smile)
        if snap.bl_result is not None:
            marginals[sym] = ContractMarginal.from_bl_result(sym, snap.bl_result, as_of=AS_OF)
    except Exception as exc:
        print(f"{sym}: {type(exc).__name__}: {str(exc)[:120]}")
print(sorted(marginals))

In [ ]:
# copula corr from the runner's forwards panel (falls back to comonotone-only)
corr = None
fwd_path = DATA_DIR / "forwards.parquet"
if fwd_path.exists():
    corr = historical_corr(pd.read_parquet(fwd_path))

snaps = run_fly_screener(marginals, TRIPLES, config=config, corr=corr, as_of=AS_OF)
monitor = screener_table(snaps)
monitor.round(3)

## 2. Deep dive: SFRU26-SFRZ26-SFRH27

Left: the fly settlement distribution `phi` with the traded entry and the median-path fly.
Right: the move-count decomposition vs the `fly/25` heuristic.

In [ ]:
deep = next(s for s in snaps if s.fly.label == "SFRU26-SFRZ26-SFRH27")
ax = plot_fly_distribution(deep)
ax = plot_move_table(deep)
c = deep.comonotone
print(f"fly {deep.fly_bp:+.2f}bp | median-path {deep.fly_median_path_bp:+.2f}bp | "
      f"tail_rent {deep.tail_rent_bp:+.2f}bp")
print(f"heuristic {deep.heuristic_prob:+.3f} vs prob_delta {c.prob_delta:+.3f}")
print("E[phi | N1-N2=j] (bp):", {j: round(v, 1) for j, v in sorted(c.e_phi_given_dn_bp.items())})
print(f"tail slopes dphi/dback: upper {c.tail_slope_upper:+.2f}, lower {c.tail_slope_lower:+.2f} bp/bp")

In [ ]:
# conditional fly by belly-rate decile (scenario axis = level of rates at the fly's pivot)
rates = comonotone_grid(deep.legs, n=20001)
phi = (2 * rates[:, 1] - rates[:, 0] - rates[:, 2]) * 100
dec = pd.qcut(rates[:, 1], 10, labels=False)
pd.DataFrame({
    "front": pd.Series(rates[:, 0]).groupby(dec).mean(),
    "belly": pd.Series(rates[:, 1]).groupby(dec).mean(),
    "back": pd.Series(rates[:, 2]).groupby(dec).mean(),
    "phi_bp": pd.Series(phi).groupby(dec).mean(),
}).round(3)

## 3. History and z-scores (from the runner's parquet)

In [ ]:
history = pd.read_parquet(DATA_DIR / "history.parquet")
print(history.groupby("label")["as_of"].agg(["count", "min", "max"]))
fig = plot_history_panel(
    history, "SFRU26-SFRZ26-SFRH27",
    cols=("fly_bp", "fly_median_path_bp", "tail_rent_bp", "tail_rent_bp_z",
          "heuristic_gap", "prob_delta_z"),
)

In [ ]:
# latest z-scores across all triples
latest = history[history["as_of"] == history["as_of"].max()].set_index("label")
zcols = [c for c in latest.columns if c.endswith("_z") or c.endswith("_z_full")]
latest[["fly_bp", "tail_rent_bp", "heuristic_gap"] + zcols].round(2)

## 4. Coupling comparison — comonotone vs Gaussian copula

The copula relaxes perfect rank correlation to the historically observed correlation of daily
contract-rate changes. The gap between the two `phi` distributions prices path-reversal risk
the comonotone model cannot see.

In [ ]:
rows = []
for s in snaps:
    row = {"label": s.fly.label,
           "como_prob_delta": s.comonotone.prob_delta,
           "como_iqr": s.comonotone.phi_quantiles_bp[75] - s.comonotone.phi_quantiles_bp[25]}
    if s.copula is not None:
        row["cop_prob_delta"] = s.copula.prob_delta
        row["cop_iqr"] = s.copula.phi_quantiles_bp[75] - s.copula.phi_quantiles_bp[25]
    rows.append(row)
pd.DataFrame(rows).set_index("label").round(3)

## 6. Optional: option-implied FOMC-path joint (expensive)

The `extract_joint` stack calibrates a common FOMC-path state library across the whites and
day-weights meetings inside reference quarters — the most honest joint model. Set
`RUN_JOINT = True` to run (minutes).

In [ ]:
from RVUtils.FlyVsVol import WingQuote, convergence_package, skew_attribution

att = skew_attribution(deep)
print(f"tail_rent {att.tail_rent_bp:+.2f} = skew_G {att.skew_g_bp:+.2f} "
      f"+ fit_residual {att.fit_residual_bp:+.2f}")
print(f"per-leg mm (mean-med): front {att.mm_front_bp:+.2f}  belly {att.mm_belly_bp:+.2f} "
      f" back {att.mm_back_bp:+.2f}   dominant: {att.dominant_leg}")

# hike-side wing quotes (price puts) straight off the cached smiles
wing_quotes = {}
for sym in deep.fly.symbols:
    smile = mdp.fetch_sabr_smile({"symbol": sym, "as_of": AS_OF,
                                  "strike_offsets_bps": "listed"})
    wing_quotes[sym] = [
        WingQuote(symbol=sym, right=p.right, strike_rate=p.strike_rate,
                  premium_bp=p.market_price * 100, delta_abs=p.delta_abs)
        for p in smile.points
        if p.right == "P" and p.market_price is not None and p.delta_abs is not None
    ]

# entry target = rolling mean of tail_rent (z-reversion); 0.0 = ride to expiry
sub = history[history["label"] == deep.fly.label].sort_values("as_of")
roll_mean = sub["tail_rent_bp"].rolling(config.zscore_window,
                                        min_periods=config.zscore_min_periods).mean().iloc[-1]
pkg = convergence_package(deep, wing_quotes, scale_lots=100,
                          target_g_bp=roll_mean, mode="curvature")
print(f"\ndirection {pkg.direction}: G {pkg.current_g_bp:+.2f}bp -> target "
      f"{pkg.target_g_bp:+.2f}bp (expected reversion {pkg.expected_reversion_bp:+.2f}bp)")
for leg in pkg.legs:
    q = leg.quote
    print(f"  {leg.side:4s} {leg.lots:4d}x {q.symbol} K_rate={q.strike_rate:.3f} "
          f"prem={q.premium_bp:.1f}bp delta={q.delta_abs:.1f} "
          f"hedge {leg.hedge_futures_lots:+.1f} futures")
print(f"net premium: {pkg.net_premium_bp_lots:+.0f} bp-lots (${pkg.net_premium_bp_lots * 25:+,.0f})")
print("skew retirement schedule (symbol, option expiry, contribution retiring):")
for sym, exp, contrib in pkg.schedule:
    print(f"  {sym}  {exp}  {contrib:+.2f}bp")

## 7. Optional: linear-curve cross-check

The screener's fly comes from the option-side futures forwards. Cross-check against the
calibrated STIR curve (same futures, curve-smoothed):

```python
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue
from TB.IRSwapsTB import IRSwapsTB
from TB.TimeseriesBuilder import TimeseriesBuilder
import pytz

NYC = pytz.timezone("America/New_York")
curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
q_fly = UnifiedQuery(curve="USD-SOFR-1D-Q12STIRT",
                     tenor="IMM_U26xIMM_Z26/IMM_Z26xIMM_H27/IMM_H27xIMM_M27",
                     value=UnifiedValue.IRS_RATE)
df = TimeseriesBuilder().get_timeseries(
    start=NYC.localize(datetime.datetime(2026, 7, 20, 17, 0)),
    end=NYC.localize(datetime.datetime(2026, 7, 27, 17, 0)),
    queries=[q_fly], freq="nyc_eod", n_jobs=4,
    routers={"IRS": IRSwapsTB(curve_mdp, show_tqdm=True)}, ignore_cache_miss=True)
df  # column: 'USD-SOFR-1D IMM_U26xIMM_Z26/IMM_Z26xIMM_H27/IMM_H27xIMM_M27 FLY RATE' (bp)
```

In [ ]:
RUN_JOINT = False
if RUN_JOINT:
    from RVUtils.ImpliedDistribution import FOMCPathStateConfig
    whites = SYMBOLS[:4]
    smiles = {s: mdp.fetch_sabr_smile({"symbol": s, "as_of": AS_OF,
                                        "strike_offsets_bps": "listed"}) for s in whites}
    current_rate = marginals[whites[0]].forward_rate
    state_cfg = FOMCPathStateConfig.default_templated_paths(current_rate=round(current_rate * 4) / 4)
    joint = dist.extract_joint(smiles, state_config=state_cfg)
    fly0 = TRIPLES[0]
    lc = joint.linear_combination_distribution(
        {fly0.front: -1.0, fly0.belly: 2.0, fly0.back: -1.0})
    print(lc.data.head(20))

## 6. Optional: linear-curve cross-check

The screener's fly comes from the option-side futures forwards. Cross-check against the
calibrated STIR curve (same futures, curve-smoothed):

```python
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue
from TB.IRSwapsTB import IRSwapsTB
from TB.TimeseriesBuilder import TimeseriesBuilder
import pytz

NYC = pytz.timezone("America/New_York")
curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
q_fly = UnifiedQuery(curve="USD-SOFR-1D-Q12STIRT",
                     tenor="IMM_U26xIMM_Z26/IMM_Z26xIMM_H27/IMM_H27xIMM_M27",
                     value=UnifiedValue.IRS_RATE)
df = TimeseriesBuilder().get_timeseries(
    start=NYC.localize(datetime.datetime(2026, 7, 20, 17, 0)),
    end=NYC.localize(datetime.datetime(2026, 7, 27, 17, 0)),
    queries=[q_fly], freq="nyc_eod", n_jobs=4,
    routers={"IRS": IRSwapsTB(curve_mdp, show_tqdm=True)}, ignore_cache_miss=True)
df  # column: 'USD-SOFR-1D IMM_U26xIMM_Z26/IMM_Z26xIMM_H27/IMM_H27xIMM_M27 FLY RATE' (bp)
```

## Caveats

- All distributions are **risk-neutral**: persistent gaps are premium, not free money.
- Comonotone cannot price V-shaped paths; its spread vs the joint measures that premium.
- SR3 settlement bins are quarter-average rates, not Fed target levels.
- Each marginal lives at its own option expiry; the coupling is an assumption on the joint law.
- Costs: fly ~0.5bp, option wings wider — sub-1bp "edges" are inside the spread.